In [1]:
!apt-get install -y rabbitmq-server > /dev/null 2>&1
!service rabbitmq-server start
!pip install -q pika
!mkdir -p stream
print("RabbitMQ Server is running!")

[ OK ]
RabbitMQ Server is running!


In [2]:
%%writefile stream/dispatcher.py
import pika, json, time, random, uuid, os

EXCHANGE_NAME = "rapido-dispatch"
RABBITMQ_URL = os.environ.get("RABBITMQ_URL", "amqp://localhost")

CITIES = ["bangalore", "chennai"]
ZONES = {
    "bangalore": ["koramangala", "indiranagar", "whitefield", "jayanagar"],
    "chennai": ["tnagar", "velachery", "adyar"]
}
TYPES = ["ride", "food"]

connection = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
channel = connection.channel()
channel.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

msg_count = 0
print("[*] Dispatcher started. Publishing every 2s...\n")

try:
    while True:
        msg_count += 1
        city = random.choice(CITIES)
        zone = random.choice(ZONES[city])
        order_type = random.choice(TYPES)
        order_id = f"ORD-{msg_count:03d}"
        ts = time.strftime("%H:%M:%S")

        # Every 10th message is intentionally corrupted (missing 'zone') for D3
        is_bad = (msg_count % 10 == 0)

        if is_bad:
            routing_key = f"{city}.unknown.{order_type}"
            body = {"id": order_id, "city": city, "type": order_type, "ts": ts}
            print(f"\n[DISPATCHER] ⚠️ DISPATCHING BAD MESSAGE #{msg_count} (missing zone) -> Key: '{routing_key}'")
        else:
            routing_key = f"{city}.{zone}.{order_type}"
            body = {"id": order_id, "city": city, "zone": zone, "type": order_type, "ts": ts}
            print(f"[DISPATCHER] Sent #{msg_count}: Key='{routing_key}' | ID={order_id}")

        channel.basic_publish(
            exchange=EXCHANGE_NAME,
            routing_key=routing_key,
            body=json.dumps(body)
        )
        time.sleep(2)
except KeyboardInterrupt:
    print("\nDispatcher stopped.")
finally:
    connection.close()

Writing stream/dispatcher.py


In [5]:
import threading, time, json, random, pika, os

EXCHANGE_NAME = "rapido-dispatch"
RABBITMQ_URL = "amqp://localhost"
stop_event = threading.Event()

# ── 1. DRIVER THREAD FUNCTION (matches driver.py) ──────────────
def run_driver(name, pattern):
    conn = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
    ch = conn.channel()
    ch.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

    q = ch.queue_declare(queue='', exclusive=True, auto_delete=True).method.queue
    ch.queue_bind(exchange=EXCHANGE_NAME, queue=q, routing_key=pattern)
    ch.basic_qos(prefetch_count=1)
    print(f"[{name}] Bound to '{pattern}'")

    def on_msg(ch, method, props, body):
        data = json.loads(body.decode())
        oid = data.get("id")
        rk = method.routing_key

        # D3 ACK / NACK validation
        if "zone" not in data or data.get("zone") is None:
            print(f"   └── [{name}] ❌ [NACK] Rejected {oid} (missing zone) | key='{rk}'")
            ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)
        else:
            print(f"   └── [{name}] ✅ [ACK] Accepted {oid} | {data['city']}.{data['zone']}.{data['type']}")
            ch.basic_ack(delivery_tag=method.delivery_tag)

    ch.basic_consume(queue=q, on_message_callback=on_msg, auto_ack=False)

    while not stop_event.is_set():
        conn.process_data_events(time_limit=0.2)
    conn.close()

# ── 2. DISPATCHER THREAD FUNCTION (matches dispatcher.py) ─────
def run_dispatcher():
    conn = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
    ch = conn.channel()
    ch.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

    CITIES = ["bangalore", "chennai"]
    ZONES = {
        "bangalore": ["koramangala", "indiranagar", "whitefield"],
        "chennai": ["tnagar"]
    }
    TYPES = ["ride", "food"]
    count = 0

    while not stop_event.is_set():
        count += 1
        city = random.choice(CITIES)
        zone = random.choice(ZONES[city])
        order_type = random.choice(TYPES)
        oid = f"ORD-{count:03d}"
        ts = time.strftime("%H:%M:%S")

        # Every 10th message is intentionally corrupted for D3
        is_bad = (count % 10 == 0)

        if is_bad:
            rk = f"{city}.unknown.{order_type}"
            body = {"id": oid, "city": city, "type": order_type, "ts": ts}
            print(f"\n[DISPATCHER] ⚠️ DISPATCHING BAD MESSAGE #{count} (missing zone) -> key='{rk}'")
        else:
            rk = f"{city}.{zone}.{order_type}"
            body = {"id": oid, "city": city, "zone": zone, "type": order_type, "ts": ts}
            print(f"\n[DISPATCHER] Sent #{count}: key='{rk}' | ID={oid}")

        ch.basic_publish(exchange=EXCHANGE_NAME, routing_key=rk, body=json.dumps(body))

        # Sleep in short slices so stop_event is responsive
        for _ in range(10):
            if stop_event.is_set(): break
            time.sleep(0.2)
    conn.close()

# ── 3. LAUNCH ALL THREADS CONCURRENTLY ─────────────────────────
stop_event.clear()
drivers = [
    ("Ravi",    "bangalore.koramangala.*"),
    ("Meena",   "bangalore.*.food"),
    ("Control", "bangalore.#")
]

threads = [threading.Thread(target=run_driver, args=(name, pat), daemon=True) for name, pat in drivers]
threads.append(threading.Thread(target=run_dispatcher, daemon=True))

for t in threads: t.start()
print("All threads running! Waiting for dispatches...\n")

try:
    time.sleep(45)  # Run for 45 seconds (or press Stop ■ anytime)
except KeyboardInterrupt:
    pass
finally:
    stop_event.set()
    time.sleep(1)
    print("\n[✓] Stopped cleanly.")

All threads running! Waiting for dispatches...


[DISPATCHER] Sent #1: key='bangalore.koramangala.food' | ID=ORD-001
[Control] Bound to 'bangalore.#'
[Meena] Bound to 'bangalore.*.food'
[Ravi] Bound to 'bangalore.koramangala.*'

[DISPATCHER] Sent #2: key='bangalore.whitefield.ride' | ID=ORD-002
   └── [Control] ✅ [ACK] Accepted ORD-002 | bangalore.whitefield.ride

[DISPATCHER] Sent #3: key='chennai.tnagar.food' | ID=ORD-003

[DISPATCHER] Sent #4: key='chennai.tnagar.ride' | ID=ORD-004

[DISPATCHER] Sent #5: key='chennai.tnagar.ride' | ID=ORD-005

[DISPATCHER] Sent #6: key='bangalore.koramangala.food' | ID=ORD-006
   └── [Ravi] ✅ [ACK] Accepted ORD-006 | bangalore.koramangala.food
   └── [Control] ✅ [ACK] Accepted ORD-006 | bangalore.koramangala.food
   └── [Meena] ✅ [ACK] Accepted ORD-006 | bangalore.koramangala.food

[DISPATCHER] Sent #7: key='chennai.tnagar.ride' | ID=ORD-007

[DISPATCHER] Sent #8: key='bangalore.whitefield.food' | ID=ORD-008
   └── [Control] ✅ [ACK] Accepted ORD-008

In [4]:
import subprocess, time, os

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Launch 3 drivers in the background
d_ravi = subprocess.Popen(["python", "-u", "stream/driver.py", "Ravi", "bangalore.koramangala.*"], env=env)
d_meena = subprocess.Popen(["python", "-u", "stream/driver.py", "Meena", "bangalore.*.food"], env=env)
d_control = subprocess.Popen(["python", "-u", "stream/driver.py", "Control", "bangalore.#"], env=env)

time.sleep(2)  # Wait for drivers to bind

# Run dispatcher
try:
    !python -u stream/dispatcher.py
except KeyboardInterrupt:
    pass
finally:
    d_ravi.terminate()
    d_meena.terminate()
    d_control.terminate()
    print("All processes stopped cleanly.")

[*] Dispatcher started. Publishing every 2s...

[DISPATCHER] Sent #1: Key='chennai.tnagar.ride' | ID=ORD-001
[DISPATCHER] Sent #2: Key='bangalore.indiranagar.ride' | ID=ORD-002
[DISPATCHER] Sent #3: Key='bangalore.koramangala.food' | ID=ORD-003
[DISPATCHER] Sent #4: Key='bangalore.whitefield.food' | ID=ORD-004
[DISPATCHER] Sent #5: Key='bangalore.jayanagar.ride' | ID=ORD-005
[DISPATCHER] Sent #6: Key='bangalore.koramangala.food' | ID=ORD-006
[DISPATCHER] Sent #7: Key='chennai.adyar.ride' | ID=ORD-007
[DISPATCHER] Sent #8: Key='bangalore.jayanagar.ride' | ID=ORD-008
[DISPATCHER] Sent #9: Key='chennai.velachery.ride' | ID=ORD-009

[DISPATCHER] ⚠️ DISPATCHING BAD MESSAGE #10 (missing zone) -> Key: 'chennai.unknown.food'
[DISPATCHER] Sent #11: Key='bangalore.whitefield.food' | ID=ORD-011
[DISPATCHER] Sent #12: Key='chennai.adyar.food' | ID=ORD-012
[DISPATCHER] Sent #13: Key='chennai.velachery.food' | ID=ORD-013
[DISPATCHER] Sent #14: Key='chennai.adyar.ride' | ID=ORD-014
[DISPATCHER] Sent 